# Counterfactual Explainer Benchmark

Benchmark counterfactual explainers across classifiers and datasets using `tscf_eval.benchmark`.

1. [Setup](#1.-Setup) — Imports, data loading, explainer and metric configuration
2. [Model Training](#2.-Model-Training) — Training for all classifiers
3. [Single Benchmark](#3.-Single-Benchmark) — One model, one dataset
4. [Multi-Model Benchmark](#4.-Multi-Model-Benchmark) — All models, one dataset
5. [Multi-Dataset Benchmark](#5.-Multi-Dataset-Benchmark) — One model, all datasets
6. [Results Analysis](#6.-Results-Analysis) — Composite scores and visualizations

## 1. Setup

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from aeon.classification.convolution_based import RocketClassifier
from aeon.classification.shapelet_based import RDSTClassifier
from aeon.classification.feature_based import TSFreshClassifier
from aeon.classification.interval_based import TimeSeriesForestClassifier
from aeon.classification.deep_learning import TimeCNNClassifier

from tscf_eval.data_loader import UCRLoader
from tscf_eval.benchmark import (
    BenchmarkRunner, BenchmarkResults, DatasetConfig, ModelConfig,
    ExplainerConfig, is_maximize,
)
from tscf_eval.counterfactuals import COMTE, NativeGuide, TSEvo, Glacier, LatentCF, SETS, CELS
from tscf_eval.evaluator import (
    Evaluator, Validity, Proximity, Sparsity, Plausibility,
    Diversity, Contiguity, Composition, Controllability, Confidence, Robustness, Efficiency,
)
from utils import evaluate_model, save_models, load_models

warnings.filterwarnings("ignore")

In [ ]:
# Global constants
DATASET_NAMES = ["ArrowHead", "CBF", "Chinatown", "ECG200", "Lightning7"]
MODEL_TYPES = ["tsf", "timecnn", "rocket", "rdst", "tsfresh_lr"]
N_COUNTERFACTUALS = 3
N_INSTANCES = 12

### Load Datasets

In [ ]:
data = {}
for name in DATASET_NAMES:
    loader = UCRLoader(name)
    train = loader.load("train")
    test = loader.load("test")
    data[name] = {
        "X_train": np.asarray(train.X),
        "y_train": np.asarray(train.y),
        "X_test": np.asarray(test.X),
        "y_test": np.asarray(test.y),
    }

print("Datasets loaded:")
for name, d in data.items():
    print(f"  {name}: train {d['X_train'].shape}, test {d['X_test'].shape}")

### Configure Explainers

In [ ]:
explainer_configs = [
    ExplainerConfig("ng_blend_dtw", NativeGuide, {"method": "blend", "distance": "dtw"}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("comte_dtw", COMTE, {"distance": "dtw", "n_counterfactuals": N_COUNTERFACTUALS}),
    ExplainerConfig("sets", SETS, {"n_shapelet_samples": 5000, "max_shapelets": 50}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("tsevo_authentic", TSEvo, {"transformer": "authentic", "n_generations": 10, "population_size": 20}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("glacier_uniform", Glacier, {"weight_type": "uniform", "max_iter": 50, "gradient_subsample": 15, "tolerance": 1e-3, "learning_rate": 0.01}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("cels", CELS, {"max_iter": 50, "tolerance": 1e-3, "gradient_subsample": 15, "learning_rate": 0.01}, n_counterfactuals=N_COUNTERFACTUALS),
]

print(f"{len(explainer_configs)} explainer variants configured")

### Configure Metrics

In [ ]:
evaluator = Evaluator([
    Validity(mode="hard"), Validity(mode="soft"),
    Proximity(p=1), Proximity(p=2), Proximity(p=float("inf")), Proximity(distance="dtw"),
    Sparsity(),
    Plausibility(method="lof"), Plausibility(method="if"), Plausibility(method="dtw_lof"),
    Diversity(distance="euclidean"), Diversity(distance="dtw"),
    Contiguity(),
    Controllability(),
    Composition(),
    Confidence(),
    Robustness(distance="euclidean"), Robustness(distance="dtw"),
    Efficiency(),
])

print(f"{len(evaluator.metrics)} metrics: {[m.name() for m in evaluator.metrics]}")

## 2. Model Training

Train all classifier types on all datasets.

| Model | Internal estimator | Smooth gradients? |
|-------|-------------------|-------------------|
| TSF | — | No (tree-based) |
| TimeCNN | CNN | Yes |
| ROCKET | RidgeClassifierCV | Yes |
| RDST | RidgeClassifierCV | Yes |
| TSFresh + LR | LogisticRegression | Yes |

In [ ]:
SEED = 42

TSC_MODEL_METRICS = [
    ("accuracy", {}),
    ("precision", {"average": "macro"}),
    ("recall", {"average": "macro"}),
    ("f1", {"average": "macro"}),
]

models = {mt: {} for mt in MODEL_TYPES}

print(f"Training {len(MODEL_TYPES)} model types x {len(DATASET_NAMES)} datasets (seed={SEED})")
print("=" * 100)

for dataset_name in DATASET_NAMES:
    d = data[dataset_name]

    print(f"\n{dataset_name} (n_train={len(d['X_train'])}):")
    print(f"{'Model':<12} {'Test F1':>10}")
    print("-" * 24)

    # ROCKET
    m = RocketClassifier(n_kernels=10000, random_state=SEED)
    m.fit(d["X_train"], d["y_train"])
    models["rocket"][dataset_name] = m
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'rocket':<12} {f1:>10.2%}")

    # RDST
    m = RDSTClassifier(max_shapelets=500, random_state=SEED)
    m.fit(d["X_train"], d["y_train"])
    models["rdst"][dataset_name] = m
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'rdst':<12} {f1:>10.2%}")

    # TSFresh + LogisticRegression
    m = TSFreshClassifier(estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=SEED), random_state=SEED)
    m.fit(d["X_train"], d["y_train"])
    models["tsfresh_lr"][dataset_name] = m
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'tsfresh_lr':<12} {f1:>10.2%}")

    # TSF
    m = TimeSeriesForestClassifier(n_estimators=200, random_state=SEED)
    m.fit(d["X_train"], d["y_train"])
    models["tsf"][dataset_name] = m
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'tsf':<12} {f1:>10.2%}")

    # TimeCNN
    m = TimeCNNClassifier(n_epochs=1000, verbose=False, random_state=SEED)
    m.fit(d["X_train"], d["y_train"])
    models["timecnn"][dataset_name] = m
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'timecnn':<12} {f1:>10.2%}")

print(f"\nTrained {len(MODEL_TYPES) * len(DATASET_NAMES)} models")

In [ ]:
# HPO summary table
summary_data = []
for mt in MODEL_TYPES:
    for ds in DATASET_NAMES:
        y_pred = models[mt][ds].predict(data[ds]["X_test"])
        test_f1 = f1_score(data[ds]["y_test"], y_pred, average="macro")
        summary_data.append({
            "Model": mt, "Dataset": ds,
            "Test F1": f"{test_f1:.2%}",
        })

pd.DataFrame(summary_data)

In [ ]:
# Save trained models (or load pre-trained ones)
# save_models(models, {mt: {ds: {"params": {}, "cv_score": 0.0} for ds in DATASET_NAMES} for mt in MODEL_TYPES}, MODEL_TYPES, DATASET_NAMES)

# To load pre-trained models instead of training:
models, _ = load_models(MODEL_TYPES, DATASET_NAMES)

## 3. Single Benchmark

Compare all explainers using **one classifier on one dataset**.
Uses RDST on ArrowHead as a representative example.

In [ ]:
SINGLE_DATASET = "ArrowHead"
SINGLE_MODEL = "rdst"

d = data[SINGLE_DATASET]

single_results = BenchmarkRunner(
    datasets=[DatasetConfig(SINGLE_DATASET, d["X_train"], d["y_train"], d["X_test"], d["y_test"])],
    models=[ModelConfig(SINGLE_MODEL, models[SINGLE_MODEL][SINGLE_DATASET])],
    explainers=explainer_configs,
    evaluator=evaluator,
    instance_selection="stratified_confidence",
    n_instances=N_INSTANCES,
    verbose=True,
).run()

with open(f"single_results_stratified_{SINGLE_MODEL}_{SINGLE_DATASET}.json", "w") as f:
    json.dump(single_results.to_dict(), f, indent=2, default=str)
print(f"Saved single_results_stratified_{SINGLE_MODEL}_{SINGLE_DATASET}.json")

In [ ]:
single_results.to_dataframe()

## 4. Multi-Model Benchmark

Compare explainers across **all classifiers on one dataset**.
This reveals how classifier choice affects counterfactual quality.

In [ ]:
MM_DATASET = "Chinatown"
d = data[MM_DATASET]

mm_dataset_config = DatasetConfig(MM_DATASET, d["X_train"], d["y_train"], d["X_test"], d["y_test"])
mm_model_configs = [ModelConfig(mt, models[mt][MM_DATASET]) for mt in MODEL_TYPES]

multi_model_results = BenchmarkRunner(
    datasets=[mm_dataset_config],
    models=mm_model_configs,
    explainers=explainer_configs,
    evaluator=evaluator,
    instance_selection="stratified_confidence",
    n_instances=N_INSTANCES,
    n_jobs=-1,
    verbose=True,
).run()

with open(f"multi_model_results_stratified_{MM_DATASET}.json", "w") as f:
    json.dump(multi_model_results.to_dict(), f, indent=2, default=str)
print(f"Saved multi_model_results_stratified_{MM_DATASET}.json")

In [ ]:
# Filter results for a specific model
SELECTED_MODEL = "rocket"  # Change to: "rocket", "rdst", "tsfresh_lr", "tsf", "timecnn"

print(f"Results for model: {SELECTED_MODEL}")
multi_model_results.filter(models=[SELECTED_MODEL]).to_dataframe()

In [ ]:
# Pivot table: compare one metric across all models
METRIC_TO_COMPARE = "validity"

df_pivot = multi_model_results.to_dataframe()
df_pivot.pivot(index="explainer", columns="model", values=METRIC_TO_COMPARE)

In [ ]:
# Aggregated statistics
print("By model:")
display(multi_model_results.aggregate(by="model"))

print("\nBy explainer:")
display(multi_model_results.aggregate(by="explainer"))

## 5. Multi-Dataset Benchmark

Compare explainers across **all datasets using one classifier per dataset**.
This identifies methods that perform consistently across different data characteristics.

In [ ]:
MD_MODEL = "rocket"

md_dataset_configs = [
    DatasetConfig(ds, data[ds]["X_train"], data[ds]["y_train"], data[ds]["X_test"], data[ds]["y_test"])
    for ds in DATASET_NAMES
]
md_model_configs = [ModelConfig(MD_MODEL, models[MD_MODEL][ds]) for ds in DATASET_NAMES]

# Run per dataset (each has its own fitted model)
multi_dataset_results = BenchmarkResults()
for ds_config, m_config in zip(md_dataset_configs, md_model_configs):
    print(f"Running {ds_config.name}...")
    for result in BenchmarkRunner(
        datasets=[ds_config], models=[m_config],
        explainers=explainer_configs,
        evaluator=evaluator,
        instance_selection="stratified_confidence",
        n_instances=N_INSTANCES,
        n_jobs=-1, verbose=False,
    ).run():
        multi_dataset_results.add(result)

print(f"\n{len(multi_dataset_results)} results collected")

with open(f"multi_dataset_results_stratified_{MD_MODEL}.json", "w") as f:
    json.dump(multi_dataset_results.to_dict(), f, indent=2, default=str)
print(f"Saved multi_dataset_results_stratified_{MD_MODEL}.json")

In [ ]:
# Filter by dataset
SELECTED_DATASET = "ArrowHead"  # Change to any dataset

print(f"Results for {SELECTED_DATASET}:")
multi_dataset_results.filter(datasets=[SELECTED_DATASET]).to_dataframe()

In [ ]:
print("By dataset:")
display(multi_dataset_results.aggregate(by="dataset"))

print("\nBy explainer (averaged across datasets):")
display(multi_dataset_results.aggregate(by="explainer"))

## 6. Results Analysis

Load combined results and compute composite scores and visualizations.

In [ ]:
# Load combined results from all seeds
df = pd.read_csv("all_results_combined.csv")
print(f"Loaded {len(df)} rows with columns: {list(df.columns[:10])}...")

### Composite Score

Weighted-sum scalarization with min-max normalization. All metrics are transformed so higher values indicate better performance.

In [ ]:
def composite_score(
    df: pd.DataFrame,
    metrics: list[str] | None = None,
    weights: dict[str, float] | None = None,
    group_by: list[str] = ["dataset", "model", "explainer"],
) -> pd.DataFrame:
    """Compute weighted composite scores with min-max normalization.
    
    Each metric is normalized to [0, 1] using min-max scaling, with all metrics
    transformed so that higher values indicate better performance. The composite
    score is a weighted sum of normalized metrics, with weights summing to one.
    
    Parameters
    ----------
    df : pd.DataFrame
        Results dataframe with metric columns.
    metrics : list[str], optional
        Metric columns to include. If None, uses default set.
    weights : dict[str, float], optional
        Per-metric weights. Automatically normalized to sum to 1.
        If None, all metrics are weighted equally.
    group_by : list[str], default ["dataset", "model", "explainer"]
        Columns to group by before computing scores.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with group columns, normalized metrics, and composite_score.
    """
    if metrics is None:
        metrics = [
            "validity_soft", "proximity_dtw", "sparsity", "plausibility_dtw_lof",
            "diversity_dpp_dtw", "contiguity", "composition", "mean_conf_cf",
            "robustness_lipschitz_dtw", "efficiency_time_s",
        ]
    
    # Aggregate across seeds
    agg_df = df.groupby(group_by, as_index=False)[metrics].mean()
    
    # Normalize weights
    if weights is None:
        weights = {m: 1.0 / len(metrics) for m in metrics}
    else:
        total = sum(weights.get(m, 0.0) for m in metrics)
        weights = {m: weights.get(m, 0.0) / total for m in metrics}
    
    # Min-max normalization respecting metric directions (from metric registry)
    normalized = agg_df[group_by].copy()
    for col in metrics:
        col_min, col_max = agg_df[col].min(), agg_df[col].max()
        rng = col_max - col_min
        if rng == 0:
            normalized[col] = 1.0
        else:
            if is_maximize(col):
                normalized[col] = (agg_df[col] - col_min) / rng
            else:
                normalized[col] = (col_max - agg_df[col]) / rng
    
    # Compute weighted composite score
    normalized["composite_score"] = sum(weights[m] * normalized[m] for m in metrics)
    
    return normalized.sort_values("composite_score", ascending=False)

In [ ]:
# Composite scores by explainer (aggregated across all datasets and models)
scores = composite_score(df, group_by=["explainer"])
scores

### Visualization Constants

In [ ]:
# Shared visualization constants
METRICS = [
    "validity_soft", "proximity_dtw", "sparsity", "plausibility_dtw_lof",
    "diversity_dpp_dtw", "contiguity", "composition", "mean_conf_cf",
    "robustness_lipschitz_dtw", "efficiency_time_s",
]

METRIC_NAMES = {
    "validity_soft": "Validity",
    "proximity_dtw": "Proximity",
    "sparsity": "Sparsity",
    "plausibility_dtw_lof": "Plausibility",
    "diversity_dpp_dtw": "Diversity",
    "contiguity": "Contiguity",
    "composition": "Composition",
    "mean_conf_cf": "Confidence",
    "robustness_lipschitz_dtw": "Robustness",
    "efficiency_time_s": "Efficiency",
}

MODEL_NAMES = {
    "timecnn": "TimeCNN",
    "tsf": "TSF",
    "rocket": "ROCKET",
    "rdst": "RDST",
    "tsfresh_lr": "TSFresh-LR",
}

EXPLAINER_NAMES = {
    "ng_blend_dtw": "Native Guide",
    "comte_dtw": "CoMTE",
    "sets": "SETS",
    "tsevo_authentic": "TSEvo",
    "cels": "CELS",
    "glacier_uniform": "Glacier",
}

EXPLAINER_ORDER = [
    "comte_dtw", "ng_blend_dtw", "tsevo_authentic",
    "sets", "cels", "glacier_uniform",
]

COLORS = {
    "comte_dtw": "#D62828",
    "ng_blend_dtw": "#003F88",
    "tsevo_authentic": "#2A9D8F",
    "sets": "#C77F20",
    "cels": "#7B2D8E",
    "glacier_uniform": "#6C757D",
}

MARKERS = {
    "comte_dtw": "o",
    "ng_blend_dtw": "s",
    "tsevo_authentic": "^",
    "sets": "D",
    "cels": "v",
    "glacier_uniform": "P",
}

### Metric Profiles by Dataset/Classifier

In [ ]:
def plot_metric_profiles(
    df: pd.DataFrame,
    group_col: str,
    name_map: dict[str, str],
    other_col: str,
) -> plt.Figure:
    """Plot normalized metric profiles grouped by *group_col*.

    Seeds are averaged first to obtain one value per (*other_col*, explainer).
    Normalization uses the full per-row range within each group.
    Shaded regions show ±1 std across *other_col*.

    Parameters
    ----------
    df : pd.DataFrame
        Raw results with columns: dataset, model, explainer, seed, + metrics.
    group_col : str
        Column whose unique values define the subplots (e.g. "dataset" or "model").
    name_map : dict
        Maps raw group values to display names.
    other_col : str
        The complementary grouping column (e.g. "model" if group_col is "dataset").

    Returns
    -------
    matplotlib.figure.Figure
    """
    groups = sorted(df[group_col].unique())
    x = np.arange(len(METRICS))

    n_cols = 3
    n_rows = (len(groups) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
    )
    axes_flat = axes.flatten()

    for idx, group in enumerate(groups):
        ax = axes_flat[idx]
        df_g = df[df[group_col] == group]

        # Step 1: average over seeds → one value per (other_col, explainer)
        seed_avg = (
            df_g.groupby([other_col, "explainer"])[METRICS]
            .mean()
            .reset_index()
        )

        # Normalization range: all individual rows in this group
        global_min = df_g[METRICS].min()
        global_max = df_g[METRICS].max()

        for explainer in EXPLAINER_ORDER:
            df_exp = seed_avg[seed_avg["explainer"] == explainer]
            if df_exp.empty:
                continue

            # Step 2: mean and std across *other_col* (not seeds)
            exp_mean = df_exp[METRICS].mean()
            exp_std = df_exp[METRICS].std()

            means, stds = [], []
            for metric in METRICS:
                rng = global_max[metric] - global_min[metric]
                if rng == 0:
                    means.append(1.0)
                    stds.append(0.0)
                else:
                    if is_maximize(metric):
                        means.append((exp_mean[metric] - global_min[metric]) / rng)
                    else:
                        means.append((global_max[metric] - exp_mean[metric]) / rng)
                    stds.append(exp_std[metric] / rng)

            means = np.array(means)
            stds = np.array(stds)

            # Shaded ±1 std band
            ax.fill_between(
                x,
                np.clip(means - stds, -0.15, 1.15),
                np.clip(means + stds, -0.15, 1.15),
                color=COLORS[explainer],
                alpha=0.10,
            )
            # Mean line
            ax.plot(
                x, means,
                marker=MARKERS[explainer],
                color=COLORS[explainer],
                linewidth=2,
                markersize=6,
                zorder=3,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(
            [METRIC_NAMES[m] for m in METRICS],
            rotation=45, ha="right", fontsize=14,
        )
        ax.set_title(name_map.get(group, group), fontsize=16, fontweight="bold")
        ax.set_ylim(-0.1, 1.1)
        ax.set_xlim(-0.4, len(METRICS) - 0.6)
        ax.grid(True, alpha=0.2, linestyle="--", linewidth=0.6)
        ax.tick_params(axis="y", labelsize=12)

        if idx % n_cols == 0:
            ax.set_ylabel("Normalized Score (↑ better)", fontsize=14)

    # Hide unused subplots
    for idx in range(len(groups), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # Shared legend
    handles = [
        Line2D([0], [0], color=COLORS[e], marker=MARKERS[e],
               markersize=12, linewidth=3, label=EXPLAINER_NAMES[e])
        for e in EXPLAINER_ORDER
    ]
    fig.legend(
        handles=handles,
        loc="lower right",
        ncol=2,
        fontsize=15,
        bbox_to_anchor=(0.96, 0.03),
        frameon=True,
        edgecolor="lightgrey",
        fancybox=True,
        columnspacing=1.5,
        handlelength=3,
    )

    fig.tight_layout()
    return fig

In [ ]:
# Figure 1: Metric profiles by dataset
dataset_map = {d: d for d in df["dataset"].unique()}
fig1 = plot_metric_profiles(df, "dataset", dataset_map, "model")
fig1.savefig("metric_profiles_by_dataset.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Figure 2: Metric profiles by classifier
fig2 = plot_metric_profiles(df, "model", MODEL_NAMES, "dataset")
fig2.savefig("metric_profiles_by_classifier.pdf", bbox_inches="tight", dpi=300)
plt.show()

### Confidence-Stratified Dumbbell Chart

In [ ]:
DUMBBELL_METRICS = {
    "validity_soft": "Validity",
    "proximity_dtw": "Proximity",
    "diversity_dpp_dtw": "Diversity",
    "mean_conf_cf": "Confidence",
    "robustness_lipschitz_dtw": "Robustness",
    "efficiency_time_s": "Efficiency",
}


def plot_quantile_dumbbell(df: pd.DataFrame) -> plt.Figure:
    """Dumbbell chart comparing Q1 (low confidence) vs Q4 (high confidence).

    For each method and metric, shows the shift from Q1 to Q4 as an arrow.
    Green = improvement, red = degradation.
    """
    df = df.copy()
    
    # Clip robustness outliers at 99th percentile
    for q in ["q1", "q2", "q3", "q4"]:
        col = f"robustness_lipschitz_dtw_{q}"
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))

    labels = [EXPLAINER_NAMES[e] for e in EXPLAINER_ORDER]

    # Compute Q1 and Q4 values per explainer
    q1_data: dict[str, dict[str, float]] = {}
    q4_data: dict[str, dict[str, float]] = {}

    for exp in EXPLAINER_ORDER:
        df_e = df[df["explainer"] == exp]
        name = EXPLAINER_NAMES[exp]
        q1_data[name], q4_data[name] = {}, {}
        for metric_base, nice_name in DUMBBELL_METRICS.items():
            q1_data[name][nice_name] = (
                df_e.groupby(["dataset", "model"])[f"{metric_base}_q1"]
                .mean().mean()
            )
            q4_data[name][nice_name] = (
                df_e.groupby(["dataset", "model"])[f"{metric_base}_q4"]
                .mean().mean()
            )

    n_methods = len(labels)
    y_pos = np.arange(n_methods)[::-1]

    fig, axes = plt.subplots(2, 3, figsize=(18, 11), sharey=True)
    axes_flat = axes.flatten()

    for ax_idx, (metric_base, nice_name) in enumerate(DUMBBELL_METRICS.items()):
        ax = axes_flat[ax_idx]
        higher_better = is_maximize(metric_base)
        
        # Add direction arrow to title
        direction_arrow = "↑" if higher_better else "↓"
        title = f"{nice_name} {direction_arrow}"

        for i, exp in enumerate(EXPLAINER_ORDER):
            name = EXPLAINER_NAMES[exp]
            v1 = q1_data[name][nice_name]
            v4 = q4_data[name][nice_name]
            y = y_pos[i]

            improved = (v4 > v1) if higher_better else (v4 < v1)
            line_color = "#2d8a4e" if improved else "#c0392b"

            # Arrow from Q1 to Q4
            ax.annotate(
                "", xy=(v4, y), xytext=(v1, y),
                arrowprops=dict(
                    arrowstyle="->,head_width=0.25,head_length=0.15",
                    color=line_color, lw=2.5, alpha=0.7,
                ),
                zorder=2,
            )
            # Q1: hollow circle
            ax.scatter(
                v1, y, s=110, color=COLORS[exp], marker="o", zorder=4,
                edgecolors=COLORS[exp], linewidths=2.2, facecolors="white",
            )
            # Q4: filled circle
            ax.scatter(v4, y, s=110, color=COLORS[exp], marker="o", zorder=4)

        ax.set_title(title, fontsize=16, fontweight="bold", pad=10)
        ax.set_yticks(y_pos)
        if ax_idx % 3 == 0:
            ax.set_yticklabels(labels, fontsize=14, fontweight="medium")
        ax.grid(axis="x", alpha=0.2, linestyle="--")
        ax.tick_params(axis="x", labelsize=12)

        # Alternating row shading
        for i in range(0, n_methods, 2):
            ax.axhspan(y_pos[i] - 0.45, y_pos[i] + 0.45, color="grey", alpha=0.05)

        # Delta annotations
        xlims = ax.get_xlim()
        x_range = xlims[1] - xlims[0]
        for i, exp in enumerate(EXPLAINER_ORDER):
            name = EXPLAINER_NAMES[exp]
            v1 = q1_data[name][nice_name]
            v4 = q4_data[name][nice_name]
            y = y_pos[i]
            dx = v4 - v1

            sign_str = f"+{dx:.2f}" if dx >= 0 else f"{dx:.2f}"
            improved = (v4 > v1) if higher_better else (v4 < v1)
            ann_color = "#2d8a4e" if improved else "#c0392b"
            ax.text(
                max(v1, v4) + x_range * 0.03, y, sign_str,
                va="center", ha="left", fontsize=10.5,
                color=ann_color, fontweight="bold",
            )
        ax.set_xlim(xlims[0] - x_range * 0.02, xlims[1] + x_range * 0.22)

    # Legend
    legend_elements = [
        Line2D([0], [0], marker="o", color="grey", markerfacecolor="white",
               markeredgecolor="grey", markersize=12, linewidth=0, label="Q1 (low confidence)"),
        Line2D([0], [0], marker="o", color="grey", markerfacecolor="grey",
               markersize=12, linewidth=0, label="Q4 (high confidence)"),
        Line2D([0], [0], color="#2d8a4e", linewidth=3, label="Improvement (Q1 → Q4)"),
        Line2D([0], [0], color="#c0392b", linewidth=3, label="Degradation (Q1 → Q4)"),
    ]
    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=4,
        fontsize=14,
        bbox_to_anchor=(0.5, -0.04),
        frameon=True,
        edgecolor="lightgrey",
        fancybox=True,
    )

    fig.tight_layout()
    return fig

In [ ]:
# Figure 3: Confidence-stratified dumbbell chart
fig3 = plot_quantile_dumbbell(df)
fig3.savefig("quantile_dumbbell.pdf", bbox_inches="tight", dpi=300)
plt.show()